### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ljubljana_primary_tumor",
    dataset_year="1987", # from the cite date on UCI...
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5WK5Q",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/83/primary+tumor.zip && unzip primary+tumor.zip primary-tumor.data && rm primary+tumor.zip && mkdir -p local-data-warehouse/ljubljana_primary_tumor && mv primary-tumor.data local-data-warehouse/ljubljana_primary_tumor/
""",
    # References
    academic_reference_bibtex="""@misc{Zwitter1987primarytumor,
  author       = {Zwitter, M. and Soklic, M.},
  title        = {{Primary Tumor}},
  year         = {1987},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5WK5Q}
}
""",
    academic_reference_bibtex_key="Zwitter1987primarytumor",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode missing values as np.nan instead of "?".
- We reverse the ordinal encoding of all features, adding the text back to the categories.
- We drop classes with less than 10 samples to have sufficient data per class for a robust benchmark task. In reality, this would call for collecting more data on these classes, or treating it as an individual few-shot prediction task.
- The data contains many duplicates (10%), which seems to be naturally occurring given the limited amount of features. Moreover, the duplicates sometimes have different targets. We keep the duplicates. While this might bias evaluation a bit, it represents the real-world task with ambiguous features better.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [ ]:
import pandas as pd

columns = [
    "class","age","sex","histologic-type","degree-of-diffe","bone","bone-marrow","lung","pleura","peritoneum","liver","brain","skin","neck","supraclavicular","axillar","mediastinum","abdominal"
]
df = pd.read_csv(dataset_mold.path / "primary-tumor.data", header=None, names=columns, na_values="?")
print("Loaded data shape:", df.shape)

domains = {
    "class": [
        "lung", "head & neck", "esophasus", "thyroid", "stomach",
        "duoden & sm.int", "colon", "rectum", "anus", "salivary glands",
        "pancreas", "gallblader", "liver", "kidney", "bladder", "testis",
        "prostate", "ovary", "corpus uteri", "cervix uteri", "vagina", "breast"
    ],
    "age": ["<30", "30-59", ">=60"],
    "sex": ["male", "female"],
    "histologic-type": ["epidermoid", "adeno", "anaplastic"],
    "degree-of-diffe": ["well", "fairly", "poorly"],
}

# yes/no attributes
yes_no_cols = [
    "bone", "bone-marrow", "lung", "pleura", "peritoneum", "liver",
    "brain", "skin", "neck", "supraclavicular", "axillar",
    "mediastinum", "abdominal"
]

for c in yes_no_cols:
    domains[c] = ["yes", "no"]

for col, vals in domains.items():
    df[col] = df[col].map({i+1: v for i, v in enumerate(vals)})

# We drop classes with less than 10 samples
df = df[df["class"].isin(df["class"].value_counts()[df["class"].value_counts() >= 10].index)]
as_cat_type = list(domains.keys())
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()